# Entrenamiento de Huellas-GAN en Colab

Fase 4 del proyecto: entrenamos la cDCGAN condicional en una GPU T4 de Colab Free.

**Estrategia de datos:**

- Código: clonamos el repo desde GitHub (rápido, siempre la última versión).
- Dataset SOCOFing: lo bajamos desde Kaggle *dentro* de Colab (necesita `kaggle.json`).
- Outputs (modelo final, samples, log): se copian a Drive al terminar para no perderlos si se reinicia la VM.

**Antes de correr:**

1. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
2. Tener el `kaggle.json` subido a tu Drive en `MyDrive/huellas-gan-secrets/kaggle.json` (o ajustar el path en la celda correspondiente).

## 1. Verificar GPU

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Mem total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('OJO: no hay GPU. Entorno de ejecución -> Cambiar tipo de entorno -> GPU.')

## 2. Clonar el repo

Lo clonamos en `/content/` (disco local de la VM, mucho más rápido que Drive).

In [ ]:
!git clone https://github.com/Arielsecchi/huellas.git /content/huellas
%cd /content/huellas/huellas-gan

## 3. Instalar dependencias

Colab ya trae `torch`, `numpy`, `matplotlib`, `opencv-python` y `tqdm`. Instalamos lo que falte (`kaggle`, etc.).

In [ ]:
!pip install -q -r requirements.txt

## 4. Montar Drive + copiar `kaggle.json`

El token de Kaggle permite bajar SOCOFing. Si todavía no tenés uno, mirá la sección "Datasets" del README.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os

KAGGLE_SRC = '/content/drive/MyDrive/huellas-gan-secrets/kaggle.json'
KAGGLE_DST = os.path.expanduser('~/.kaggle/kaggle.json')

os.makedirs(os.path.dirname(KAGGLE_DST), exist_ok=True)
shutil.copy(KAGGLE_SRC, KAGGLE_DST)
os.chmod(KAGGLE_DST, 0o600)
print('kaggle.json listo en', KAGGLE_DST)

## 5. Descargar SOCOFing + preproceso + etiquetado Vucetich

Las tres fases del pipeline de datos corren en <10 min totales. El resultado (`data/processed/images.npz` y `data/processed/metadata.csv`) queda en el disco local de la VM.

In [ ]:
!python -m src.data.download_socofing

In [ ]:
!python -m src.data.preprocess

In [ ]:
!python -m src.data.label_vucetich --full

## 6. Smoke test del training loop (2 iteraciones)

Chequea que todo enchufa antes del entrenamiento largo.

In [ ]:
from src.training.config import TrainConfig
from src.training.train import train

train(TrainConfig(epochs=1, batch_size=16, num_workers=0,
                  max_steps=2, sample_every_epochs=1, ckpt_every_epochs=1))

## 7. Entrenamiento completo

50 épocas con batch 64 tardan ~30-40 min en una T4. Si se corta la sesión, los checkpoints quedan en `/content/huellas/huellas-gan/models/checkpoints/` — se pierden al reiniciar la VM, pero el modelo final se copia a Drive en la celda 8.

In [ ]:
cfg = TrainConfig(
    epochs=50,
    batch_size=64,
    num_workers=2,
    sample_every_epochs=2,
    ckpt_every_epochs=10,
)
train(cfg)

## 8. Persistir outputs a Drive

Copiamos a `MyDrive/huellas-gan-outputs/`:

- `generator.pt` (el modelo final — lo usa la app de Fase 6)
- `training_samples/` (grillas por época + log CSV)

Los checkpoints intermedios NO los copiamos (pesan mucho). Si los querés persistir, agregá la línea correspondiente.

In [ ]:
import shutil
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/huellas-gan-outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

shutil.copy('models/final/generator.pt', DRIVE_OUT / 'generator.pt')
if (DRIVE_OUT / 'training_samples').exists():
    shutil.rmtree(DRIVE_OUT / 'training_samples')
shutil.copytree('outputs/training_samples', DRIVE_OUT / 'training_samples')
print('outputs copiados a', DRIVE_OUT)

## 9. Ver samples y curva de pérdida

In [ ]:
from IPython.display import Image, display
from pathlib import Path

samples = sorted(Path('outputs/training_samples').glob('epoch_*.png'))
print(f'{len(samples)} grillas de samples')
if samples:
    display(Image(filename=str(samples[-1])))

In [ ]:
import csv
import matplotlib.pyplot as plt

with open('outputs/training_samples/train_log.csv') as f:
    rows = list(csv.DictReader(f))
epochs = [int(r['epoch']) for r in rows]
loss_d = [float(r['loss_d']) for r in rows]
loss_g = [float(r['loss_g']) for r in rows]

plt.figure(figsize=(8, 4))
plt.plot(epochs, loss_d, label='D')
plt.plot(epochs, loss_g, label='G')
plt.xlabel('época')
plt.ylabel('loss (media de la época)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Listo

El modelo final quedó en `MyDrive/huellas-gan-outputs/generator.pt`. En Fase 5 lo cargamos para evaluar (FID, clasificación Vucetich automática sobre samples), y en Fase 6 lo usa la app web para generar huellas on-demand.